# Notebook 3 — Janelamento e Extração de Features

Terceiro estágio do pipeline. Lê os arquivos `_L1.npz` e produz, para cada paciente, nível de canais e partição (treino/teste), a matriz de features usada na classificação.

## Balanceamento das classes

O conjunto de treino é balanceado em proporção 1:1 entre pré-ictal e interictal; o conjunto de teste preserva a distribuição natural.

| Partição | Pré-ictal | Interictal | Justificativa |
|----------|-----------|------------|---------------|
| Treino | N janelas | N janelas amostradas | evita viés do classificador para a classe majoritária |
| Teste | todas | todas | distribuição realista, com métricas robustas a desbalanceamento |

No treino, dadas N janelas pré-ictais, amostram-se N janelas interictais distantes de crises, com distribuição estratificada no tempo de gravação (uma janela por faixa temporal) para refletir variabilidade circadiana. No teste não há reamostragem.

## Níveis de canais

A redução progressiva de eletrodos simula configurações compatíveis com dispositivos vestíveis.

| Nível | Canais | Quantidade | Datasets |
|-------|--------|------------|----------|
| R5 | fp1,fp2,f7,f3,fz,f4,f8,t3,t4,t5,t6,c3,c4,p3,p4 | 15 | CHB-MIT, Siena, Mendeley |
| R3 | f7,f3,fz,f4,f8,t3,t4,t5,t6,c3,c4 | 11 | CHB-MIT, Siena, Mendeley |
| R2 | f7,f3,fz,f4,f8,t3,t4,t5,t6 | 9 | CHB-MIT, Siena, Mendeley |
| R1 | t3,t4,t5,t6 | 4 | CHB-MIT, Siena, Mendeley |
| R0 | t3,t4 | 2 | CHB-MIT, Siena, Mendeley, SeizeIT2 |

## Features por canal

| Grupo | Features | Quantidade |
|-------|----------|------------|
| Temporais | desvio-padrão, variância, RMS, comprimento de linha, mobilidade de Hjorth, assimetria, curtose | 7 |
| Espectrais | potência em delta, theta, alpha, beta, gamma; entropia espectral; potência relativa em beta | 7 |
| Wavelet (DWT) | energia dos coeficientes de detalhe d1–d4 | 4 |
| Não-linear | complexidade de Hjorth | 1 |
| Total | | 19 |


## 1. Dependências e configuração

In [1]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

%pip install -q numpy pandas matplotlib scipy PyWavelets seaborn tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re, glob, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch
from scipy.stats import skew, kurtosis
import pywt
from tqdm.auto import tqdm
import IPython.display as ipd

import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')

# Compatibilidade NumPy >= 2.0
if not hasattr(np, 'trapz'):
    np.trapz = np.trapezoid

ROOT_DIR   = 'data'
SIGNAL_DIR = os.path.join(ROOT_DIR, 'signals')
L1_DIR     = os.path.join(ROOT_DIR, 'level1_signals')
FEAT_DIR = os.path.join(ROOT_DIR, 'features')
LOG_DIR  = os.path.join(ROOT_DIR, 'logs'); os.makedirs(LOG_DIR, exist_ok=True)
for lv in ['R5', 'R3', 'R2', 'R1', 'R0']:
    os.makedirs(os.path.join(FEAT_DIR, lv, 'train'), exist_ok=True)
    os.makedirs(os.path.join(FEAT_DIR, lv, 'test'), exist_ok=True)

LBL = dict(interictal=0, preictal=1, ictal=2, postictal=3, unknown=-1)

# Janelamento
WIN_SEC = 30; STEP_TRAIN_SEC = 30; STEP_TEST_SEC = 15; MIN_PURITY = 0.80
# Balanceamento (interictal:preictal no treino)
BALANCE_RATIO_TRAIN = 1

BANDS = {'delta':(0.5,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,40)}

FEAT_NAMES = [
    'std','var','rms','line_len','mobility','skewness','kurtosis',
    'delta','theta','alpha','beta','gamma','sp_entropy','beta_rel',
    'dwt_d1','dwt_d2','dwt_d3','dwt_d4','complexity',
]
N_FEAT = len(FEAT_NAMES)
assert N_FEAT == 19

PATIENTS = {
    'CHBMIT'  : ['chb01','chb03','chb04','chb05','chb06','chb07','chb08','chb10','chb11','chb12','chb13','chb14'],
    'Siena'   : ['PN01','PN03','PN05','PN06','PN07','PN09','PN10','PN11','PN12','PN13','PN14','PN16'],
    'Mendeley': ['p10','p11','p12','p13','p14','p15'],
    'SeizeIT2': ['sub-001','sub-002','sub-003','sub-004','sub-005','sub-006','sub-007','sub-008','sub-009','sub-010','sub-011','sub-012'],
}
CHANNEL_SETS = {
    'R5': ['fp1','fp2','f7','f3','fz','f4','f8','t3','t4','t5','t6','c3','c4','p3','p4'],
    'R3': ['f7','f3','fz','f4','f8','t3','t4','t5','t6','c3','c4'],
    'R2': ['f7','f3','fz','f4','f8','t3','t4','t5','t6'],
    'R1': ['t3','t4','t5','t6'],
    'R0': ['t3','t4'],
}
LEVEL_DS = {
    'R5': ['CHBMIT','Siena','Mendeley'],
    'R3': ['CHBMIT','Siena','Mendeley'],
    'R2': ['CHBMIT','Siena','Mendeley'],
    'R1': ['CHBMIT','Siena','Mendeley'],
    'R0': ['CHBMIT','Siena','Mendeley','SeizeIT2'],
}
# Equivalência de nomenclatura entre datasets
CH_RENAME = {
    'CHBMIT': {'t7':'t3','t8':'t4','p7':'t5','p8':'t6'},
    'SeizeIT2': {'bteleftsd':'t3','bterightsd':'t4','crosstopsd':'t3'},
}
print('Configuração carregada.')
print(f'Janela {WIN_SEC} s | passo treino {STEP_TRAIN_SEC} s | passo teste {STEP_TEST_SEC} s')
print(f'Balanceamento de treino {BALANCE_RATIO_TRAIN}:1 (interictal:preictal)')
for lv, chs in CHANNEL_SETS.items():
    print(f'  {lv}: {len(chs):2d} canais x {N_FEAT} = {len(chs)*N_FEAT:4d} features | {", ".join(LEVEL_DS[lv])}')

Configuração carregada.
Janela 30 s | passo treino 30 s | passo teste 15 s
Balanceamento de treino 1:1 (interictal:preictal)
  R5: 15 canais x 19 =  285 features | CHBMIT, Siena, Mendeley
  R3: 11 canais x 19 =  209 features | CHBMIT, Siena, Mendeley
  R2:  9 canais x 19 =  171 features | CHBMIT, Siena, Mendeley
  R1:  4 canais x 19 =   76 features | CHBMIT, Siena, Mendeley
  R0:  2 canais x 19 =   38 features | CHBMIT, Siena, Mendeley, SeizeIT2


c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Extração de features

`channel_feats` calcula as 19 features de um canal, na ordem de `FEAT_NAMES`. `build_fvec` concatena as features de todos os canais do nível; canais ausentes em um dataset são preenchidos com zeros, mantendo a dimensionalidade fixa.


In [3]:
def normalize_ch(name, rename_map=None):
    s = str(name).lower()
    for sub in ['eeg', '-ref', '-le', '-a1', '-a2', 'ref']:
        s = s.replace(sub, ' ')
    s = s.strip().replace(' ', '')
    if '-' in s:
        s = s.split('-')[0]
    s = re.sub(r'[^a-z0-9]', '', s)
    return rename_map.get(s, s) if rename_map else s

def channel_feats(sig, sfreq=256.0):
    '''19 features de um canal, na ordem de FEAT_NAMES.'''
    if len(sig) < 2:
        return np.zeros(N_FEAT, dtype=np.float32)
    d1 = np.diff(sig); d2 = np.diff(d1) if len(d1) > 1 else np.array([0.0])
    act = float(np.var(sig)); mob = float(np.sqrt(np.var(d1) / (act + 1e-10)))
    mob2 = float(np.sqrt(np.var(d2) / (np.var(d1) + 1e-10)))
    temporal = [float(np.std(sig)), act, float(np.sqrt(np.mean(sig**2))),
                float(np.sum(np.abs(d1))), mob, float(skew(sig)), float(kurtosis(sig))]
    nperseg = min(int(sfreq), max(int(sfreq // 2), len(sig) // 2)); nperseg = max(min(nperseg, len(sig)), 2)
    f, psd = welch(sig, fs=sfreq, nperseg=nperseg); total = psd.sum() + 1e-10
    bp = []
    for lo, hi in BANDS.values():
        idx = (f >= lo) & (f <= hi)
        bp.append(float(np.trapz(psd[idx], f[idx])) if idx.sum() > 1 else 0.0)
    pn = psd / total; pn = pn[pn > 0]; sp_ent = float(-np.sum(pn * np.log(pn)))
    b_idx = (f >= 13) & (f <= 30)
    beta_rel = float(np.trapz(psd[b_idx], f[b_idx]) / total) if b_idx.sum() > 1 else 0.0
    spectral = bp + [sp_ent, beta_rel]
    ml = min(4, pywt.dwt_max_level(len(sig), 'db4')) if len(sig) > 1 else 0
    if ml >= 1:
        coeffs = pywt.wavedec(sig.astype(np.float64), 'db4', level=ml)
        dwt = [float(np.sum(c**2)) for c in coeffs[1:5]]
        while len(dwt) < 4:
            dwt.append(0.0)
    else:
        dwt = [0.0] * 4
    complexity = mob2 / (mob + 1e-10)
    feat = np.array(temporal + spectral + dwt + [complexity], dtype=np.float32)
    assert len(feat) == N_FEAT
    return feat

def build_fvec(win_data, ch_names_raw, dataset, level):
    '''Concatena as features dos canais do nível; ausentes preenchidos com zeros.'''
    rename = CH_RENAME.get(dataset, {})
    ch_map = {normalize_ch(c, rename): i for i, c in enumerate(ch_names_raw)}
    parts = []
    for tgt in CHANNEL_SETS[level]:
        i = ch_map.get(tgt)
        parts.append(channel_feats(win_data[i].astype(np.float64)) if i is not None
                     else np.zeros(N_FEAT, dtype=np.float32))
    return np.concatenate(parts)

print('Funções de extração de features definidas.')
_t = channel_feats(np.random.randn(256 * 30).astype(np.float32), 256.0)
print(f'Verificação: channel_feats retornou {len(_t)} features.')

Funções de extração de features definidas.
Verificação: channel_feats retornou 19 features.


## 3. Janelamento

Cada janela de 30 s recebe a classe dominante (no mínimo `MIN_PURITY` das amostras). Apenas janelas interictais e pré-ictais são mantidas; ictal e pós-ictal não fazem parte da tarefa de predição. Para cada janela interictal registra-se o instante de início, usado na amostragem estratificada.


In [4]:
def extract_windows(dataset, patient, level, mode):
    '''Extrai janelas de todas as gravações de um paciente.
       Retorna (X_pre, X_inter, t_inter, n_arquivos).

    O sinal é carregado arquivo por arquivo e liberado imediatamente após
    o janelamento (del data + gc.collect). O pico de RAM é o maior arquivo
    individual do paciente.
    '''
    step_sec     = STEP_TRAIN_SEC if mode == 'train' else STEP_TEST_SEC
    files        = sorted(glob.glob(os.path.join(L1_DIR, f'{dataset}__{patient}__*_L1.npz')))
    n_feat_total = len(CHANNEL_SETS[level]) * N_FEAT
    X_pre, X_inter, t_inter = [], [], []
    global_t = 0.0

    for fp in files:
        # Carrega metadados leves do _L1.npz (sem sinal)
        try:
            z      = np.load(fp, allow_pickle=True)
            labels = z['labels']
            sfreq  = float(z['sfreq'])
            chs    = list(z['ch_names'])
            n      = int(z['n_samples'])
            z.close(); del z
        except Exception as e:
            tqdm.write(f'  {os.path.basename(fp)}: {e}'); continue

        # Carrega sinal do _signal.npz original
        sig_path = fp.replace(L1_DIR, SIGNAL_DIR).replace('_L1.npz', '_signal.npz')
        try:
            zs   = np.load(sig_path, allow_pickle=True)
            data = zs['data']          # float32, pico de RAM aqui
            zs.close(); del zs
        except Exception as e:
            tqdm.write(f'  {os.path.basename(sig_path)}: {e}')
            del labels; continue

        win_n  = int(WIN_SEC   * sfreq)
        step_n = max(1, int(step_sec * sfreq))

        for start in range(0, n - win_n + 1, step_n):
            wl    = labels[start:start + win_n]
            valid = wl[wl >= 0]
            if len(valid) == 0:
                continue
            vals, counts = np.unique(valid, return_counts=True)
            dom = vals[np.argmax(counts)]
            if dom not in (LBL['interictal'], LBL['preictal']):
                continue
            if counts.max() / win_n < MIN_PURITY:
                continue
            fv = build_fvec(data[:, start:start + win_n], chs, dataset, level)
            if dom == LBL['preictal']:
                X_pre.append(fv)
            else:
                X_inter.append(fv)
                t_inter.append(global_t + start / sfreq)

        global_t += n / sfreq
        del data, labels; gc.collect()   # libera sinal imediatamente

    return (np.array(X_pre,   dtype=np.float32) if X_pre   else np.empty((0, n_feat_total), dtype=np.float32),
            np.array(X_inter, dtype=np.float32) if X_inter else np.empty((0, n_feat_total), dtype=np.float32),
            np.array(t_inter, dtype=np.float64),
            len(files))

print('Função de janelamento definida.')

Função de janelamento definida.


## 4. Amostragem estratificada e montagem do conjunto

No treino, as N janelas interictais são amostradas dividindo o tempo de gravação em N faixas e tomando uma janela por faixa, o que evita concentração temporal. No teste, todas as janelas são mantidas.


In [5]:
def stratified_sample(X_inter, t_inter, n_target, seed=42):
    '''Amostra n_target janelas interictais distribuídas no tempo.'''
    n = len(X_inter)
    if n <= n_target:
        return X_inter
    rng = np.random.default_rng(seed)
    order = np.argsort(t_inter)
    bins = np.array_split(order, n_target)
    pick = [rng.choice(b) for b in bins if len(b) > 0]
    pick = np.array(pick[:n_target])
    return X_inter[np.sort(pick)]

def make_dataset(dataset, patient, level, mode):
    '''Monta (X, y). Treino: 1:1. Teste: distribuição natural.'''
    X_pre, X_inter, t_inter, n_files = extract_windows(dataset, patient, level, mode)
    n_pre = len(X_pre); n_inter_all = len(X_inter)
    if n_pre == 0:
        return None, None, dict(reason='sem janelas pré-ictais', n_files=n_files,
                                n_pre=0, n_inter=n_inter_all)
    if mode == 'train':
        n_target = n_pre * BALANCE_RATIO_TRAIN
        X_inter_s = stratified_sample(X_inter, t_inter, n_target, seed=42)
    else:
        X_inter_s = X_inter
    if len(X_inter_s) == 0:
        return None, None, dict(reason='sem janelas interictais', n_files=n_files,
                                n_pre=n_pre, n_inter=0)
    X = np.concatenate([X_inter_s, X_pre])
    y = np.concatenate([np.zeros(len(X_inter_s), dtype=np.int8),
                        np.ones(n_pre, dtype=np.int8)])
    info = dict(reason=None, n_files=n_files, n_pre=n_pre,
                n_inter_all=n_inter_all, n_inter_used=len(X_inter_s),
                ratio=round(len(X_inter_s) / max(n_pre, 1), 2))
    return X, y, info

def save_features(X, y, dataset, patient, level, mode):
    fname = os.path.join(FEAT_DIR, level, mode, f'{dataset}__{patient}.npz')
    np.savez_compressed(fname, X=X, y=y,
        feat_names=np.array(FEAT_NAMES), channels=np.array(CHANNEL_SETS[level]),
        dataset=np.str_(dataset), patient=np.str_(patient),
        level=np.str_(level), mode=np.str_(mode),
        win_sec=np.float32(WIN_SEC),
        step_sec=np.float32(STEP_TRAIN_SEC if mode == 'train' else STEP_TEST_SEC),
        n_feat_per_ch=np.int32(N_FEAT))
    return fname

print('Funções de amostragem e montagem definidas.')

Funções de amostragem e montagem definidas.


## 5. Extração

Percorre níveis, datasets, pacientes e partições. A gravação é incremental, permitindo retomada. Exclusões (paciente sem pré-ictal ou sem interictal em um nível) são registradas em log.


In [6]:
EXCLUSION_LOG = []; STATS = []

def run_extraction(dataset_filter=None):
    pending = []
    for lv, ds_list in LEVEL_DS.items():
        for ds in ds_list:
            if dataset_filter and ds != dataset_filter:
                continue
            for pat in PATIENTS[ds]:
                for mode in ['train', 'test']:
                    out = os.path.join(FEAT_DIR, lv, mode, f'{ds}__{pat}.npz')
                    if not os.path.exists(out):
                        pending.append((lv, ds, pat, mode, out))
    if not pending:
        print('Nada pendente.'); return
    print(f'Extraindo {len(pending)} tarefas.')
    pbar = tqdm(pending, desc='Extração', ncols=110)
    done = warn = 0
    for (lv, ds, pat, mode, out) in pbar:
        pbar.set_postfix_str(f'{lv}|{ds}|{pat}|{mode}')
        if os.path.exists(out):
            continue
        X, y, info = make_dataset(ds, pat, lv, mode)
        if X is None:
            EXCLUSION_LOG.append(dict(level=lv, dataset=ds, patient=pat, mode=mode, **info))
            tqdm.write(f'  excluído {lv}/{mode}/{ds}/{pat}: {info["reason"]}'); warn += 1; continue
        save_features(X, y, ds, pat, lv, mode)
        STATS.append(dict(level=lv, dataset=ds, patient=pat, mode=mode,
                          n_total=len(X), n_pre=int((y == 1).sum()),
                          n_inter=int((y == 0).sum()), ratio=info['ratio']))
        done += 1
        if mode == 'train':
            tqdm.write(f'  {lv}/{mode}/{ds}/{pat}: {len(X)} janelas '
                       f'(pré={info["n_pre"]}, inter={info["n_inter_used"]})')
    print(f'Extração concluída: {done} gravados | {warn} excluídos.')

run_extraction()

Extraindo 324 tarefas.


Extração:   0%|                                       | 1/324 [00:40<3:36:05, 40.14s/it, R5|CHBMIT|chb01|test]

  R5/train/CHBMIT/chb01: 314 janelas (pré=157, inter=157)


Extração:   1%|▎                                      | 3/324 [02:35<4:35:19, 51.46s/it, R5|CHBMIT|chb03|test]

  R5/train/CHBMIT/chb03: 240 janelas (pré=120, inter=120)


...

  R0/train/SeizeIT2/sub-011: 156 janelas (pré=78, inter=78)


Extração: 100%|████████████████████████████████▉| 323/324 [3:26:10<00:51, 51.00s/it, R0|SeizeIT2|sub-012|test]

  R0/train/SeizeIT2/sub-012: 160 janelas (pré=80, inter=80)


Extração: 100%|█████████████████████████████████| 324/324 [3:26:58<00:00, 38.33s/it, R0|SeizeIT2|sub-012|test]

Extração concluída: 324 gravados | 0 excluídos.


## 6. Resumo e exclusões

In [7]:
if STATS:
    df_stats = pd.DataFrame(STATS)
    print('Treino (balanceado 1:1) por dataset e nível:')
    tr = df_stats[df_stats['mode'] == 'train'].groupby(['level', 'dataset']).agg(
        pacientes=('patient', 'count'), janelas=('n_total', 'sum'),
        pre=('n_pre', 'sum'), inter=('n_inter', 'sum')).reset_index()
    ipd.display(tr)
    print('\nTeste (distribuição natural) por dataset e nível:')
    te = df_stats[df_stats['mode'] == 'test'].groupby(['level', 'dataset']).agg(
        pacientes=('patient', 'count'), janelas=('n_total', 'sum'),
        pre=('n_pre', 'sum'), inter=('n_inter', 'sum')).reset_index()
    te['razao_real'] = (te['inter'] / te['pre']).round(1)
    ipd.display(te)
    df_stats.to_csv(os.path.join(LOG_DIR, 'features_stats.csv'), index=False)

if EXCLUSION_LOG:
    df_exc = pd.DataFrame(EXCLUSION_LOG)
    print(f'\nExclusões: {len(df_exc)}')
    ipd.display(df_exc)
    df_exc.to_csv(os.path.join(LOG_DIR, 'exclusoes_nb3.csv'), index=False)

print('\nExtração concluída. Próximo: Notebook 3.5 (verificação) e Notebook 4 (cenários).')

Treino (balanceado 1:1) por dataset e nível:


,level,dataset,pacientes,janelas,pre,inter
0,R0,CHBMIT,12,3698,1849,1849
1,R0,Mendeley,6,1597,870,727
2,R0,SeizeIT2,12,2012,1006,1006
3,R0,Siena,12,1870,935,935
4,R1,CHBMIT,12,3698,1849,1849
5,R1,Mendeley,6,1597,870,727
6,R1,Siena,12,1870,935,935
7,R2,CHBMIT,12,3698,1849,1849
8,R2,Mendeley,6,1597,870,727
9,R2,Siena,12,1870,935,935



Teste (distribuição natural) por dataset e nível:


,level,dataset,pacientes,janelas,pre,inter,razao_real
0,R0,CHBMIT,12,50718,3689,47029,12.7
1,R0,Mendeley,6,4402,1740,2662,1.5
2,R0,SeizeIT2,12,194740,2019,192721,95.5
3,R0,Siena,12,17471,1866,15605,8.4
4,R1,CHBMIT,12,50718,3689,47029,12.7
5,R1,Mendeley,6,4402,1740,2662,1.5
6,R1,Siena,12,17471,1866,15605,8.4
7,R2,CHBMIT,12,50718,3689,47029,12.7
8,R2,Mendeley,6,4402,1740,2662,1.5
9,R2,Siena,12,17471,1866,15605,8.4



Extração concluída. Próximo: Notebook 3.5 (verificação) e Notebook 4 (cenários).
